In [ ]:
# Run once in a fresh notebook environment.
%pip install -q numpy scipy pandas matplotlib


# Gaussian Heuristic Regimes

**Purpose.** Reproduce final-choice and information-acquisition comparisons, one-dimensional parameter sweeps, targeted 50/50 and equal-outcome searches, and approximation diagnostics in the continuous Gaussian model.

This notebook evaluates the current Gaussian implementation with 1,200 episodes, 500 VOI samples, common true states, and common observation streams. For the identified historical broad and focused grid recipes, use [Historical Gaussian Grid Reference](historical_gaussian_grids.ipynb): the older terminal solver and chunk settings differ. Current-model reanalysis must not be labelled exact reproduction of those older tables. Set `RUN = True` only where the full computation should start.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys


def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "scripts").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook from inside the repository checkout.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
print(PROJECT_ROOT)


In [ ]:
RUN = False
MODE = "full"  # smoke checks wiring; full retains the reported settings.
assert MODE in ("smoke", "full")
PRESET = "server" if MODE == "full" else "smoke"
EPISODES = 1200 if MODE == "full" else 12
VOI_SAMPLES = 500 if MODE == "full" else 24
OBSERVATIONS_PER_PERSON = 500 if MODE == "full" else 24
MAX_WORKERS = max(1, (os.cpu_count() or 2) - 1)
OUTPUT_ROOT = "results/heuristic_mapping/gaussian_regimes/" + MODE

print({"run": RUN, "episodes": EPISODES, "voi_samples": VOI_SAMPLES, "max_workers": MAX_WORKERS})


## Comparisons, Sweeps, DP, and Gauss-Hermite

This command generates final-choice and information-acquisition tables, behavior profiles, approximation-method comparisons, one-dimensional sweeps, DP sensitivity results, and Gauss-Hermite diagnostics.


In [ ]:
main_command = [
    sys.executable, str(PROJECT_ROOT / "scripts" / "run_parallel_experiments.py"),
    "--preset", PRESET,
    "--sections", "step7,sweeps,dp,gh",
    "--episodes", str(EPISODES),
    "--voi-samples", str(VOI_SAMPLES),
    "--common-observations", "on",
    "--observations-per-person", str(OBSERVATIONS_PER_PERSON),
    "--max-workers", str(MAX_WORKERS),
    "--output-dir", OUTPUT_ROOT + "/main",
]
print(" ".join(main_command))
if RUN:
    subprocess.run(main_command, cwd=PROJECT_ROOT, check=True)


## Targeted regime searches

The grids separately search for near-50/50 allocation, symmetric equal-outcome behavior, and equal-outcome behavior that is distinct from equal division.


In [ ]:
GRID_NAMES = ["near_50_50_focused", "equal_outcome_focused", "equal_outcome_distinct_focused"]
GRID_CHUNKS = 32
for grid_name in GRID_NAMES:
    command = [
        sys.executable, str(PROJECT_ROOT / "scripts" / "run_parallel_experiments.py"),
        "--preset", PRESET,
        "--sections", "regime_grid",
        "--regime-grid", grid_name,
        "--regime-grid-chunks", str(GRID_CHUNKS),
        "--episodes", str(EPISODES),
        "--voi-samples", str(VOI_SAMPLES),
        "--common-observations", "on",
        "--observations-per-person", str(OBSERVATIONS_PER_PERSON),
        "--max-workers", str(MAX_WORKERS),
        "--output-dir", OUTPUT_ROOT + "/" + grid_name,
    ]
    print(" ".join(command))
    if RUN:
        subprocess.run(command, cwd=PROJECT_ROOT, check=True)


## Inspect outputs

The runner combines each section's CSVs at its output root and generates SVG summaries. The previews below read those combined outputs, never an arbitrary shard. Missing files do not trigger computation. Inspect all comparison, sweep, candidate, DP and Gauss-Hermite outputs listed in the notebook index; negative and near-zero regimes remain part of the evidence.


In [ ]:
import pandas as pd

output_root = PROJECT_ROOT / OUTPUT_ROOT
for directory in (output_root / "main", *(output_root / name for name in GRID_NAMES)):
    for path in sorted(directory.glob("*.csv")):
        print(path.relative_to(output_root))
        display(pd.read_csv(path).head())
